# **1. Importing Necessary Libraries**

In [18]:
# Import necessary libraries
import numpy as np
import pandas as pd
import scipy.io as sio
import joblib
import json
import os
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# ML and Deep Learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, ConfusionMatrixDisplay

# Tensorflow
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, regularizers

# **2. Data Loading**

In [22]:
elec = sio.loadmat('dataset_elec.mat')
amb = sio.loadmat('dataset_amb.mat')

In [23]:
# Extract arrays
vdc1 = elec['vdc1'].flatten()
vdc2 = elec['vdc2'].flatten()
idc1 = elec['idc1'].flatten()
idc2 = elec['idc2'].flatten()
irr = amb['irr'].flatten()
temp = amb['pvt'].flatten()
label = amb['f_nv'].flatten()

In [25]:
# Create dataframe
df = pd.DataFrame({
    'vdc1': vdc1,
    'vdc2': vdc2,
    'idc1': idc1,
    'idc2': idc2,
    'irr': irr,
    'temp': temp,
    'label': label
})

In [26]:
# Number of samples
print("Number of samples:", len(df))

Number of samples: 1373798


In [27]:
df.label.value_counts()

,count
label,
0,1162931
4,188473
2,10371
3,6024
1,5999


In [28]:
# Remove degradation
df = df[df['label'] != 4].reset_index(drop=True)
print("Number of samples after removing degradation:", len(df))

Number of samples after removing degradation: 1185325


# **3. Data Preprocessing**

In [29]:
# Create power features
df['power1'] = df['vdc1'] * df['idc1']
df['power2'] = df['vdc2'] * df['idc2']
df['total_power']  = df['power1'] + df['power2']

In [31]:
df['v_diff'] = df['vdc1'] - df['vdc2']
df['i_diff'] = df['idc1'] - df['idc2']
df['p_diff'] = df['power1'] - df['power2']

In [32]:
# Rate change features
df['vdc1_change'] = df['vdc1'].diff().fillna(0)
df['vdc2_change'] = df['vdc2'].diff().fillna(0)
df['idc1_change'] = df['idc1'].diff().fillna(0)
df['idc2_change'] = df['idc2'].diff().fillna(0)

In [34]:
feats = [col for col in df.columns if col != 'label']

In [36]:
# Seperate data
X = df.drop('label', axis=1).values
y = df['label'].values

In [37]:
# Encoding
ohe = OneHotEncoder()
y = ohe.fit_transform(y.reshape(-1, 1)).toarray()

In [38]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

In [40]:
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, stratify=y_train, random_state=42)

In [39]:
# Scale features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [41]:
print("Training set:", len(X_train),"samples")
print("Validation set:", len(X_val),"samples")
print("Testing set:", len(X_test),"samples")

Training set: 758608 samples
Validation set: 189652 samples
Testing set: 237065 samples
